# 06 — Diffusion Transformer (DiT)

**Paper:** *Scalable Diffusion Models with Transformers* (Peebles & Xie, NeurIPS 2023)  
**arXiv:** https://arxiv.org/abs/2212.09748

---

## Motivation: Why Replace U-Net?

Latent Diffusion Models (LDM / Stable Diffusion) use a **U-Net** as their denoising backbone.  
U-Net was designed for semantic segmentation in 2015 — it works, but it comes with hard-coded inductive biases:
- Fixed encoder–decoder topology with skip connections
- Convolution-heavy design limits scalability

**DiT** asks: *can we replace U-Net with a plain Vision Transformer?*

Answer: **Yes** — and it scales better. DiT follows the classic finding that Transformer scaling laws (more parameters → better loss) hold for image generation too.

## DiT Architecture Overview

DiT takes a **latent patch** as input (from a VAE encoder), applies Transformer blocks conditioned on the diffusion timestep and class label, then predicts the noise.

![DiT Architecture](./figures/DiT_arch.png)

*Figure: DiT processes latent image patches with Transformer blocks conditioned via adaLN-Zero.*

## How DiT Works: Step by Step

```
Latent z  (B, C, H/8, W/8)           ← from VAE encoder
    │
    ▼  patchify  (p×p patches, e.g. p=2)
Tokens  (B, N, d_model)              ← N = (H/8 * W/8) / p²
    +  positional encoding (sinusoidal 2D)
    │
    ▼  conditioning
Timestep t  →  sinusoidal embed  →  MLP  →  c_t   (B, d_model)
Class label y  →  embedding table   →  c_y  (B, d_model)
c = c_t + c_y                                (or concatenate)
    │
    ▼  DiT Block × L
  adaLN-Zero(c): scale/shift LayerNorm by learned linear(c)
  Multi-Head Self-Attention
  adaLN-Zero(c): scale/shift LayerNorm
  Feed-Forward Network (GELU, 4× expansion)
    │
    ▼  Linear decoder
Predict noise ε̂  and  log-variance Σ̂
    (B, N, p²·C)   →  unpatchify  →  (B, C, H/8, W/8)
```

### adaLN-Zero (key conditioning mechanism)

Adaptive Layer Norm **Zero** initialises each DiT block's output to the identity at the start of training:

```python
# Each block: given conditioning vector c
scale, shift = linear(c).chunk(2, dim=-1)
x = LayerNorm(x) * (1 + scale) + shift
# + extra gate α → initialised to 0 so block starts as identity
```

This makes early training stable — the network acts as if the Transformer blocks aren't there yet.

## Four Block Designs

DiT paper compares four ways to inject the conditioning signal:

![DiT Block Designs](./figures/dit_block.png)

| Design | How conditioning enters |
|--------|------------------------|
| **In-Context** | Append `c_t`, `c_y` as extra tokens (like ViT's CLS) |
| **Cross-Attention** | Cross-attend to conditioning tokens after self-attention |
| **adaLN** | Predict scale + shift for LayerNorm from `c` |
| **adaLN-Zero** | Same + zero-init the residual gate → **best** |

**adaLN-Zero** is the winner — lowest FID with fewest extra parameters.

In [ ]:
# pip install torch torchvision einops matplotlib
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# ── Sinusoidal timestep embedding (identical to DDPM / ViT pos-enc) ──

def sinusoidal_embedding(t, dim):
    """
    t: (B,) integer timesteps  or  (B, 1)
    Returns: (B, dim) sinusoidal embedding
    """
    assert dim % 2 == 0
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(half, device=t.device) / (half - 1)
    )           # (half,)
    t = t.float().view(-1, 1)          # (B, 1)
    args = t * freqs.unsqueeze(0)      # (B, half)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)


# Quick sanity check
t_test = torch.arange(8)
emb = sinusoidal_embedding(t_test, 256)
print("Timestep embedding shape:", emb.shape)  # (8, 256)

plt.figure(figsize=(10, 3))
plt.imshow(emb.numpy().T, aspect='auto', cmap='RdBu')
plt.colorbar()
plt.title("Sinusoidal Timestep Embeddings  (rows=dim, cols=timestep)")
plt.xlabel("Timestep"); plt.ylabel("Embedding dim")
plt.tight_layout(); plt.show()

In [ ]:
# ── adaLN-Zero DiT Block ──

class DiTBlock(nn.Module):
    """
    Single DiT transformer block with adaLN-Zero conditioning.
    
    conditioning vector c: (B, d_model)
    Predicts 6 parameters per token position:
        γ1, β1  → scale/shift before self-attention
        α1      → gate on attention residual
        γ2, β2  → scale/shift before FFN
        α2      → gate on FFN residual
    All initialised to 0 so block starts as identity map.
    """

    def __init__(self, d_model, n_heads, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, elementwise_affine=False)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model, elementwise_affine=False)
        mlp_dim = int(d_model * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, mlp_dim), nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(mlp_dim, d_model), nn.Dropout(drop)
        )
        # adaLN-Zero: output 6 * d_model scalars from conditioning vector
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(d_model, 6 * d_model, bias=True)
        )
        # Zero-init so all gates start at 0 → block is identity at init
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def modulate(self, x, shift, scale):
        return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

    def forward(self, x, c):
        # c: (B, d_model)
        g1, s1, a1, g2, s2, a2 = self.adaLN_modulation(c).chunk(6, dim=-1)
        # Self-attention with adaLN
        h = self.modulate(self.norm1(x), s1, g1)
        attn_out, _ = self.attn(h, h, h)
        x = x + a1.unsqueeze(1) * attn_out
        # FFN with adaLN
        h = self.modulate(self.norm2(x), s2, g2)
        x = x + a2.unsqueeze(1) * self.ffn(h)
        return x


# Test a single block
block = DiTBlock(d_model=384, n_heads=6)
x_test = torch.randn(2, 16, 384)   # (B, N_tokens, d_model)
c_test = torch.randn(2, 384)        # (B, d_model)
out = block(x_test, c_test)
print("DiTBlock output:", out.shape)  # (2, 16, 384)

In [ ]:
# ── Full DiT Model ──

class DiT(nn.Module):
    """
    Diffusion Transformer for class-conditional image generation.
    
    Input:  noisy latent x  (B, C, H, W)
            timestep        (B,)
            class label     (B,)  integer in [0, num_classes)
    Output: predicted noise (B, C, H, W)   [or noise + log-var]
    """

    def __init__(
        self,
        in_channels=4,      # latent channels (VAE output)
        patch_size=2,
        d_model=384,
        depth=12,
        n_heads=6,
        num_classes=1000,
        mlp_ratio=4.0,
        learn_sigma=True,   # predict noise + log-variance
    ):
        super().__init__()
        self.patch_size  = patch_size
        self.in_channels = in_channels
        self.learn_sigma = learn_sigma
        out_channels = in_channels * 2 if learn_sigma else in_channels

        # Patch embedding: (B, C, H, W) → (B, N, d_model)
        self.patch_embed = nn.Conv2d(
            in_channels, d_model,
            kernel_size=patch_size, stride=patch_size, bias=True
        )

        # Conditioning: timestep + class
        self.time_embed = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.SiLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.class_embed = nn.Embedding(num_classes + 1, d_model)  # +1 for unconditional

        # Transformer blocks
        self.blocks = nn.ModuleList([
            DiTBlock(d_model, n_heads, mlp_ratio) for _ in range(depth)
        ])

        # Final norm + linear decoder
        self.final_norm = nn.LayerNorm(d_model, elementwise_affine=False)
        self.final_adaLN = nn.Sequential(nn.SiLU(), nn.Linear(d_model, 2 * d_model))
        self.final_linear = nn.Linear(d_model, patch_size * patch_size * out_channels)

        # Positional encoding (learned)
        self.pos_embed = None  # will be set on first forward

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
        nn.init.zeros_(self.final_linear.weight)
        nn.init.zeros_(self.final_linear.bias)

    def _get_pos_embed(self, H, W):
        # 2D sinusoidal positional embedding
        grid_h = torch.arange(H, dtype=torch.float32)
        grid_w = torch.arange(W, dtype=torch.float32)
        grid = torch.meshgrid(grid_h, grid_w, indexing='ij')
        grid = torch.stack(grid, dim=0)                   # (2, H, W)
        d = self.patch_embed.out_channels // 2
        freq = torch.exp(-math.log(10000) * torch.arange(d // 2) / (d // 2 - 1))
        pe_h = torch.cat([torch.sin(grid[0:1] * freq.view(-1,1,1)),
                           torch.cos(grid[0:1] * freq.view(-1,1,1))], dim=0)  # (d, H, W)
        pe_w = torch.cat([torch.sin(grid[1:2] * freq.view(-1,1,1)),
                           torch.cos(grid[1:2] * freq.view(-1,1,1))], dim=0)
        pe = torch.cat([pe_h, pe_w], dim=0)               # (d_model, H, W)
        return pe.flatten(1).T.unsqueeze(0)                # (1, H*W, d_model)

    def unpatchify(self, x, H, W):
        # x: (B, N, p²·C) → (B, C, H*p, W*p)
        p = self.patch_size
        C = self.in_channels * (2 if self.learn_sigma else 1)
        x = x.reshape(x.shape[0], H, W, p, p, C)
        x = x.permute(0, 5, 1, 3, 2, 4).contiguous()
        return x.reshape(x.shape[0], C, H * p, W * p)

    def forward(self, x, t, y):
        # x: (B, C, H, W),  t: (B,) timestep,  y: (B,) class index
        B, C, H, W = x.shape
        pH, pW = H // self.patch_size, W // self.patch_size

        # Patchify
        tokens = self.patch_embed(x)                       # (B, d, pH, pW)
        tokens = tokens.flatten(2).transpose(1, 2)         # (B, pH*pW, d)

        # Add positional encoding
        pos = self._get_pos_embed(pH, pW).to(x.device)    # (1, N, d)
        tokens = tokens + pos

        # Build conditioning vector
        t_emb = sinusoidal_embedding(t, tokens.shape[-1]).to(x.device)
        c = self.time_embed(t_emb) + self.class_embed(y)   # (B, d)

        # Transformer blocks
        for block in self.blocks:
            tokens = block(tokens, c)

        # Final adaLN + linear
        shift, scale = self.final_adaLN(c).chunk(2, dim=-1)
        tokens = self.final_norm(tokens) * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
        tokens = self.final_linear(tokens)                  # (B, N, p²·out_C)

        # Unpatchify back to spatial
        out = self.unpatchify(tokens, pH, pW)               # (B, out_C, H, W)
        return out


# ── Model variants (from paper Table 1) ──
DiT_configs = {
    "DiT-S/2": dict(patch_size=2, d_model=384, depth=12, n_heads=6),
    "DiT-B/4": dict(patch_size=4, d_model=768, depth=12, n_heads=12),
    "DiT-L/4": dict(patch_size=4, d_model=1024, depth=24, n_heads=16),
    "DiT-XL/2": dict(patch_size=2, d_model=1152, depth=28, n_heads=16),
}

model = DiT(**DiT_configs["DiT-S/2"], num_classes=10, learn_sigma=False).to(device)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"DiT-S/2  parameters: {total_params:.1f}M")

# Forward pass test — 8×8 latent (= 64×64 pixel image with VAE stride 8)
x_fake = torch.randn(2, 4, 8, 8).to(device)
t_fake = torch.randint(0, 1000, (2,)).to(device)
y_fake = torch.randint(0, 10,   (2,)).to(device)
out_fake = model(x_fake, t_fake, y_fake)
print(f"Input:  {x_fake.shape}   Output: {out_fake.shape}")

## Scaling: DiT Outperforms U-Net as Model Grows

One of the key results in the paper is that DiT follows **scaling laws** — larger models consistently achieve lower FID scores.

![DiT GFLOPs vs FID](./figures/dit_gflops.png)

*Figure: FID-50K on ImageNet 256×256 vs training compute (GFLOPs). DiT-XL/2 achieves FID = 2.27, surpassing all prior diffusion models.*

| Model | Params | FID-50K (256×256) |
|-------|--------|-------------------|
| DiT-S/2 | 33M | 68.4 |
| DiT-B/4 | 130M | 43.5 |
| DiT-L/4 | 458M | 23.5 |
| DiT-XL/2 | 675M | **2.27** |

Compare to ADM (U-Net based): FID = 10.94 with 554M params.

## Block Design Ablation

![DiT Block Comparison](./figures/dit_adaLN.png)

*Figure: adaLN-Zero conditioning achieves the best FID vs GFLOPs among all four conditioning strategies.*

The zero-initialisation of the residual gate is crucial — it means at the beginning of training, the Transformer block is a perfect identity function, making gradient flow stable from step 0.

In [ ]:
# ── DDPM Noise Schedule + DiT Training Step ──

class DDPMSchedule:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02):
        self.T = T
        betas = torch.linspace(beta_start, beta_end, T)
        alphas = 1.0 - betas
        alphas_bar = torch.cumprod(alphas, dim=0)
        self.register = {
            "betas":         betas,
            "alphas":        alphas,
            "alphas_bar":    alphas_bar,
            "sqrt_abar":     alphas_bar.sqrt(),
            "sqrt_1m_abar":  (1 - alphas_bar).sqrt(),
        }

    def q_sample(self, x0, t):
        """Forward diffusion: add noise to x0 at timestep t"""
        noise = torch.randn_like(x0)
        sa  = self.register["sqrt_abar"][t].view(-1, 1, 1, 1).to(x0.device)
        s1a = self.register["sqrt_1m_abar"][t].view(-1, 1, 1, 1).to(x0.device)
        return sa * x0 + s1a * noise, noise


schedule = DDPMSchedule(T=1000)

# ── Single training step ──
def training_step(model, schedule, x0, y):
    """
    x0: (B, 4, H, W) clean latents
    y:  (B,) class labels
    Returns: MSE loss on noise prediction
    """
    B = x0.shape[0]
    t = torch.randint(0, schedule.T, (B,), device=x0.device)
    xt, noise = schedule.q_sample(x0, t)

    noise_pred = model(xt, t, y)
    loss = F.mse_loss(noise_pred, noise)
    return loss


# Demo: one step (no actual training, just shapes)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.0)

x0_demo = torch.randn(4, 4, 8, 8).to(device)
y_demo  = torch.randint(0, 10, (4,)).to(device)

loss = training_step(model, schedule, x0_demo, y_demo)
loss.backward()
optimizer.step()
optimizer.zero_grad()

print(f"Training step loss: {loss.item():.4f}")

In [ ]:
# ── Classifier-Free Guidance (CFG) ──
# DiT uses CFG at inference: blend conditional and unconditional predictions
# This requires training with random label dropout (10% of samples use y=num_classes as NULL)

def cfg_forward(model, xt, t, y, guidance_scale=4.0, null_label=10):
    """
    Classifier-free guidance inference pass.
    guidance_scale=1.0 → pure conditional
    guidance_scale>1.0 → stronger conditioning (more class-faithful, less diverse)
    """
    B = xt.shape[0]
    null_y = torch.full_like(y, null_label)

    # Single forward pass with both conditional and unconditional (batch them together)
    xt_double = torch.cat([xt, xt], dim=0)
    t_double  = torch.cat([t,  t],  dim=0)
    y_double  = torch.cat([y,  null_y], dim=0)

    model.eval()
    with torch.no_grad():
        pred = model(xt_double, t_double, y_double)

    cond, uncond = pred.chunk(2, dim=0)
    guided = uncond + guidance_scale * (cond - uncond)
    return guided


# Demo CFG
model_cfg = DiT(**DiT_configs["DiT-S/2"], num_classes=10, learn_sigma=False).to(device)
xt = torch.randn(2, 4, 8, 8).to(device)
t  = torch.randint(0, 1000, (2,)).to(device)
y  = torch.randint(0, 10, (2,)).to(device)

guided_pred = cfg_forward(model_cfg, xt, t, y, guidance_scale=4.0, null_label=10)
print(f"CFG output shape: {guided_pred.shape}")   # (2, 4, 8, 8)

# Show effect of guidance scale
scales = [1.0, 2.0, 4.0, 7.5]
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, gs in zip(axes, scales):
    pred = cfg_forward(model_cfg, xt, t, y, guidance_scale=gs, null_label=10)
    arr = pred[0, 0].cpu().detach().numpy()
    im = ax.imshow(arr, cmap='RdBu', vmin=-3, vmax=3)
    ax.set_title(f"scale={gs}", fontsize=12)
    ax.axis('off')
fig.colorbar(im, ax=axes, fraction=0.02)
plt.suptitle("Effect of Classifier-Free Guidance Scale on Predicted Noise", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── DDPM Reverse Sampling with DiT ──

@torch.no_grad()
def ddpm_sample(model, schedule, shape, y, guidance_scale=4.0, null_label=10, device="cpu"):
    """
    Ancestral sampling: start from pure noise, denoise T steps.
    shape: (B, C, H, W)  — latent shape
    """
    B, C, H, W = shape
    x = torch.randn(B, C, H, W, device=device)

    betas     = schedule.register["betas"].to(device)
    alphas    = schedule.register["alphas"].to(device)
    alphas_bar = schedule.register["alphas_bar"].to(device)

    for i in reversed(range(schedule.T)):
        t_batch = torch.full((B,), i, device=device, dtype=torch.long)

        # Predict noise with CFG
        eps = cfg_forward(model, x, t_batch, y, guidance_scale, null_label)

        # DDPM update
        alpha_t    = alphas[i]
        alpha_bar_t = alphas_bar[i]
        beta_t     = betas[i]

        coef = beta_t / (1 - alpha_bar_t).sqrt()
        x0_pred = (x - coef * eps) / alpha_bar_t.sqrt()
        x0_pred = x0_pred.clamp(-1, 1)

        if i > 0:
            alpha_bar_prev = alphas_bar[i - 1]
            posterior_var = beta_t * (1 - alpha_bar_prev) / (1 - alpha_bar_t)
            noise = torch.randn_like(x)
            x = (alpha_bar_prev.sqrt() * beta_t / (1 - alpha_bar_t)) * x0_pred +                 ((1 - alpha_bar_prev) * alpha_t.sqrt() / (1 - alpha_bar_t)) * x +                 posterior_var.sqrt() * noise
        else:
            x = x0_pred

    return x


# Demo sampling — generate 4 samples (untrained model, just checks shapes)
print("Running toy sampling (T=20 for speed demo)...")
schedule_short = DDPMSchedule(T=20, beta_start=0.01, beta_end=0.2)
y_sample = torch.zeros(4, dtype=torch.long).to(device)
samples = ddpm_sample(model_cfg, schedule_short, (4, 4, 8, 8), y_sample,
                      guidance_scale=4.0, null_label=10, device=device)
print(f"Generated latent shape: {samples.shape}")   # (4, 4, 8, 8)
print("Mean: {:.3f},  Std: {:.3f}".format(samples.mean().item(), samples.std().item()))

## DiT vs U-Net: Key Differences

| | U-Net (ADM) | DiT |
|--|-------------|-----|
| **Backbone** | Convolutional encoder-decoder | Pure Transformer (ViT-like) |
| **Conditioning** | Feature-map AdaGN | adaLN-Zero on token sequence |
| **Attention** | Spatial attention in bottleneck | Global self-attention all layers |
| **Scaling** | Inductive biases limit scalability | Follows Transformer scaling laws |
| **Positional info** | Built-in via conv locality | 2D sinusoidal / learned pos-enc |
| **FID (256×256)** | 10.94 (ADM-G, 554M) | **2.27** (DiT-XL/2, 675M) |

### Where DiT is used today

- **Stable Diffusion 3** (2024): Multi-Modal Diffusion Transformer (MMDiT)
- **FLUX** (Black Forest Labs, 2024): DiT with flow matching
- **Sora** (OpenAI, 2024): Video DiT — spatial + temporal patches
- **PixArt-α/Σ** (2023/2024): Efficient DiT for text-to-image

## Summary

| Component | Details |
|-----------|---------|
| **Backbone** | ViT — patchify latent → token sequence |
| **Patch size** | p=2 or p=4 on the latent space (already 8× compressed by VAE) |
| **Conditioning** | adaLN-Zero: predict scale, shift, gate from `c = t_emb + class_emb` |
| **Training objective** | MSE on ε-prediction (same as DDPM) |
| **Guidance** | Classifier-Free Guidance (null token for unconditional) |
| **Scaling result** | FID improves monotonically with model size + compute |

### Key Equations

**Forward diffusion** (same as DDPM):
$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar\alpha_t} x_0, (1-\bar\alpha_t)\mathbf{I})$$

**adaLN-Zero modulation**:
$$x \leftarrow x + \alpha \cdot \text{Block}\!\left(\text{LN}(x) \cdot (1+\gamma) + \beta\right), \quad [\gamma, \beta, \alpha] = \text{Linear}(c)$$

**CFG at inference**:
$$\hat\epsilon = \epsilon_\theta(x_t, \varnothing) + s \cdot \left(\epsilon_\theta(x_t, y) - \epsilon_\theta(x_t, \varnothing)\right)$$

### References

| Paper | Link |
|-------|------|
| Scalable Diffusion Models with Transformers (DiT) | [arxiv 2212.09748](https://arxiv.org/abs/2212.09748) |
| Stable Diffusion 3 (MMDiT) | [arxiv 2403.03206](https://arxiv.org/abs/2403.03206) |
| FLUX — Flow Matching with DiT | [github.com/black-forest-labs/flux](https://github.com/black-forest-labs/flux) |
| Sora — Video generation as a world simulator | [openai.com/sora](https://openai.com/sora) |